# DataStax Astra DB Vector Store with Gemini Embeddings and Groq

## What this lesson builds

This notebook connects **DataStax Astra DB Serverless** to LangChain. Astra DB stores and searches vectors remotely; **Gemini generates the embeddings**; **Groq generates natural-language answers** in the optional RAG section.

The instructor notebook's core sequence is preserved: create/connect to Astra DB → configure embeddings → create `AstraDBVectorStore` → insert the ten example documents → similarity search → metadata filtering → score inspection → retriever → score-threshold retriever → MMR retriever. A clearly labelled extension then connects the retriever to Groq with LCEL.

### Provider separation

- **Gemini-only cells** configure and test embedding generation. These cells do not initialize Groq.
- **Astra DB cells** connect to the database and insert/query documents. They do not create an LLM.
- **Groq-only cells** configure and test answer generation. They do not call Gemini.
- The final RAG chain is the intentional integration point: it passes retrieved documents from Astra DB to Groq.

### Credential warning

The supplied instructor notebook contained credential-like strings in code cells. Do not reuse or republish those values. Revoke/rotate any real exposed Astra DB or OpenAI credentials, and use your own keys via environment variables or Colab Secrets. This revised notebook contains no hard-coded secrets.

## 1. Course map and prerequisites

**Vector database** means the remote database service that stores records and supports vector search. **Vector store** is the application/library interface that lets our code add documents, create embeddings and retrieve similar documents. In this lesson, `AstraDBVectorStore` is the LangChain adapter, and Astra DB is the remote backend.

You need an Astra DB Serverless database configured for vector search, its Data API endpoint, and an application token. For the local embedding path used here, you also need a Gemini API key. The Groq API key is needed only for the optional generation/RAG portion.

**Why bring our own embeddings?** This makes the embedding provider explicit and keeps the pipeline portable: the client-side Gemini model creates vectors, and Astra DB stores those vectors. Astra's optional server-side Vectorize integrations are a different setup and are not used here.

Official references:
- Astra DB + LangChain: https://docs.datastax.com/en/astra-db-serverless/integrations/langchain.html
- `AstraDBVectorStore` API: https://reference.langchain.com/python/langchain-astradb/vectorstores/AstraDBVectorStore
- Gemini embeddings: https://ai.google.dev/gemini-api/docs/embeddings
- LangChain Groq: https://python.langchain.com/docs/integrations/chat/groq/

## 2. VS Code setup (Windows + `uv`)

Open a Windows CMD terminal in your project folder. These commands install the dependencies into the project's `uv` environment and add Jupyter support:

```cmd
cd /d E:\RAG
uv add langchain-core langchain-astradb google-genai langchain-groq python-dotenv
uv add --dev ipykernel jupyter
uv run jupyter lab
```

You can also open this `.ipynb` directly in VS Code using the Jupyter extension, then choose the Python interpreter/kernel belonging to this `uv` project. **Do not use `uv run requirements.txt`**; `uv run` executes a command, it does not install a requirements file. If you already have a project `pyproject.toml`, do not run `uv init` again.

Your current secrets file is `E:\RAG\.env` (one folder above this notebook's directory). Fill it with your own values locally; never commit `.env` to Git. The configuration cell explicitly loads that file. If you move the notebook or `.env`, update `env_file` in Cell 4.

### 2.1 Load local environment variables

**What/why:** `.env` stores configuration outside source code. This notebook explicitly loads your current file at `E:\RAG\.env` because the notebook is open from a nested folder (`E:\RAG\Vector_stores_&_Vector_databases`). If you move the project, update `env_file` in the next cell. Secret values are never printed.


In [22]:
import os
from pathlib import Path
from dotenv import load_dotenv

# This is the .env path you are using in this project.
env_file = Path(r"E:\RAG\.env")

print("Working directory:", Path.cwd())
print(".env exists:", env_file.is_file())

if not env_file.is_file():
    raise FileNotFoundError(
        f".env file not found at: {env_file}. "
        "Update env_file if your .env lives somewhere else."
    )

# Reload this exact file so the notebook uses the values you just edited.
load_dotenv(env_file, override=True)

print("Loaded environment variables from:", env_file)

Working directory: e:\RAG\Vector_stores_&_Vector_databases
.env exists: True
Loaded environment variables from: E:\RAG\.env


## 3. Shared LangChain and Astra imports

These are framework/data imports only. Gemini and Groq SDKs are imported later in separate provider-specific cells. `Document` is LangChain's container for text plus metadata; `AstraDBVectorStore` is the interface to the remote Astra collection.

In [23]:
import os
from typing import Sequence

from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from langchain_astradb import AstraDBVectorStore

## 4. Astra DB configuration (VS Code)

In the DataStax Astra DB console, create a **Serverless Vector** database for this tutorial (for practice, choose the vector-optimized serverless option, then choose an appropriate cloud provider and region). After creation, copy the Data API endpoint from the database connection details. In account settings, open the tokens section and generate an application token with only the permissions needed for this database. Put these values in `.env` as shown in `.env.example`. For Colab, place them in Colab Secrets. After insertion, you can use Astra DB Data Explorer to inspect the collection and filter records by metadata such as `source = tweet` or `source = news`; the database dashboard can show usage/throughput separately from the Python query results.

**Why a dedicated collection name?** A collection has a fixed vector dimension and metric. This lesson uses 768-dimensional Gemini vectors, so it uses a fresh, clearly named collection rather than the instructor's OpenAI collection. If you change embedding model/dimensions later, use a new collection or deliberately rebuild the existing collection.

In [24]:
ASTRA_DB_API_ENDPOINT = (
    os.getenv("ASTRA_DB_API_ENDPOINT", "").strip().rstrip("/")
)
ASTRA_DB_APPLICATION_TOKEN = (
    os.getenv("ASTRA_DB_APPLICATION_TOKEN", "").strip()
)
# This is a keyspace/namespace, NOT the database UUID or database name.
ASTRA_DB_NAMESPACE = (
    os.getenv("ASTRA_DB_NAMESPACE", "default_keyspace").strip()
    or "default_keyspace"
)
ASTRA_DB_COLLECTION_NAME = (
    os.getenv(
        "ASTRA_DB_COLLECTION_NAME",
        "astra_gemini_groq_tutorial_768d",
    ).strip()
    or "astra_gemini_groq_tutorial_768d"
)

if not ASTRA_DB_API_ENDPOINT or not ASTRA_DB_APPLICATION_TOKEN:
    raise ValueError(
        "Set ASTRA_DB_API_ENDPOINT and ASTRA_DB_APPLICATION_TOKEN in E:\\RAG\\.env."
    )

print("Endpoint loaded:", bool(ASTRA_DB_API_ENDPOINT))
print("Token loaded:", bool(ASTRA_DB_APPLICATION_TOKEN))
print("Namespace:", ASTRA_DB_NAMESPACE)
print("Collection:", ASTRA_DB_COLLECTION_NAME)

Endpoint loaded: True
Token loaded: True
Namespace: default_keyspace
Collection: gemini_groq_documents


## 4.1 Verify Astra DB connectivity before embedding work

**What/why:** This is a small connection check that runs before Gemini embedding requests or collection creation. It verifies that the endpoint and token can list collections in the selected keyspace.

An output of `Collections: []` is **not an error**. It means authentication worked but no collections exist in that keyspace yet. The `AstraDBVectorStore` initialization cell later in this notebook is expected to create the tutorial collection if needed.

Keep the endpoint as the base URL copied from Astra DB Connection details; do not append `/api/json/v1/<database-name>` yourself. `ASTRA_DB_NAMESPACE` is a keyspace name, not the database UUID. If you did not create a custom keyspace, use `default_keyspace`.


In [25]:
from astrapy import DataAPIClient

if not ASTRA_DB_API_ENDPOINT or not ASTRA_DB_APPLICATION_TOKEN:
    raise ValueError("Endpoint or application token is missing.")

client = DataAPIClient()
database = client.get_database(
    ASTRA_DB_API_ENDPOINT,
    token=ASTRA_DB_APPLICATION_TOKEN,
    keyspace=ASTRA_DB_NAMESPACE,
)

# This request proves that Astra DB accepts the endpoint/token pair.
existing_collections = database.list_collection_names()
print("Astra DB connection successful.")
print("Existing collections:", existing_collections)

if not existing_collections:
    print(
        "No collections exist in this keyspace yet. This is expected for a new database; "
        "the AstraDBVectorStore cell below will create the tutorial collection."
    )

Astra DB connection successful.
Existing collections: ['gemini_groq_documents']


## 5. Gemini embeddings — provider setup only

**Definition:** An embedding is a numeric vector representing text in a learned semantic space. Similar meanings can map to nearby vectors, enabling semantic retrieval even if the query and document use different wording.

**Why Gemini here?** We use Gemini only for document/query vector generation. The adapter implements LangChain's `Embeddings` interface so Astra DB can call the same object for indexing and querying.

This notebook uses `gemini-embedding-2` with 768 output dimensions. For text retrieval, Google's current guide recommends formatting documents as `title: ... | text: ...` and search queries as `task: search result | query: ...`. Use the same model, dimensions and format consistently for a collection.

### 5.1 Configure the Gemini API key

Add `GEMINI_API_KEY` to your local `.env`. The cell below configures Gemini only; it does not initialize Groq.

In [26]:
from google import genai
from google.genai import types

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
if not GEMINI_API_KEY:
    raise ValueError("Set GEMINI_API_KEY in .env before running this cell.")

gemini_client = genai.Client(api_key=GEMINI_API_KEY)
GEMINI_EMBEDDING_MODEL = "gemini-embedding-2"
EMBEDDING_DIMENSIONS = 768
print("Gemini client configured:", bool(gemini_client))
print("Embedding model:", GEMINI_EMBEDDING_MODEL)
print("Requested dimensions:", EMBEDDING_DIMENSIONS)

Gemini client configured: True
Embedding model: gemini-embedding-2
Requested dimensions: 768


### 5.2 Implement a LangChain-compatible Gemini adapter

**What:** `embed_documents()` embeds the documents being stored; `embed_query()` embeds a user query. These methods are separate because the model's recommended search formatting differs for documents and queries.

**Why embed one text per API call here?** It keeps the teaching implementation easy to inspect and avoids confusing `gemini-embedding-2`'s multi-input aggregation behavior with a batch of independent vectors. For production ingestion, add controlled batching, retry/backoff, rate-limit handling and caching.

In [27]:
class GeminiEmbeddings(Embeddings):
    """Small teaching adapter for Gemini Embedding 2 and LangChain."""

    def __init__(self, client, model: str, output_dimensionality: int = 768):
        self.client = client
        self.model = model
        self.output_dimensionality = output_dimensionality

    def _embed_one(self, text: str) -> list[float]:
        response = self.client.models.embed_content(
            model=self.model,
            contents=text,
            config=types.EmbedContentConfig(
                output_dimensionality=self.output_dimensionality
            ),
        )
        if not response.embeddings or not response.embeddings[0].values:
            raise RuntimeError("Gemini returned no embedding values.")
        return list(response.embeddings[0].values)

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return [self._embed_one(f"title: none | text: {text}") for text in texts]

    def embed_query(self, text: str) -> list[float]:
        return self._embed_one(f"task: search result | query: {text}")


embeddings = GeminiEmbeddings(
    client=gemini_client,
    model=GEMINI_EMBEDDING_MODEL,
    output_dimensionality=EMBEDDING_DIMENSIONS,
)
print("LangChain Gemini embedding adapter created.")

LangChain Gemini embedding adapter created.


### 5.3 Test Gemini embeddings before touching Astra DB

**Why:** Test each layer independently. If this cell fails, debug Gemini authentication/model access before diagnosing Astra DB.

This makes a live API request and may consume quota. It prints only the vector dimension and a short numeric preview, not the API key.

In [28]:
sample_query_vector = embeddings.embed_query("What is the weather forecast?")
print("Vector dimensions:", len(sample_query_vector))
print("First 8 values:", sample_query_vector[:8])
assert len(sample_query_vector) == EMBEDDING_DIMENSIONS

Vector dimensions: 768
First 8 values: [0.057800774, 0.024254842, 0.0256095, -0.008087319, -0.0028578923, 0.048057906, 0.0037729633, 0.0051932232]


## 6. Connect LangChain to Astra DB

**What:** `AstraDBVectorStore` is the LangChain vector-store adapter. It uses the Gemini embedding adapter defined above and sends the resulting vectors, text and metadata to the remote Astra DB collection.

**Why now?** The independent connection test has already verified that the endpoint and token work. If it printed `Existing collections: []`, that is expected before this step. The next cell initializes `ASTRA_DB_COLLECTION_NAME`; AstraDBVectorStore may create the collection automatically if it does not already exist.

The collection must match the embedding dimension (768 here). Do not point this tutorial at a collection built with another model or a different vector dimension.


In [29]:
vector_store = AstraDBVectorStore(
    embedding=embeddings,
    api_endpoint=ASTRA_DB_API_ENDPOINT,
    token=ASTRA_DB_APPLICATION_TOKEN,
    collection_name=ASTRA_DB_COLLECTION_NAME,
    namespace=ASTRA_DB_NAMESPACE,
    metric="cosine",
)
print("AstraDBVectorStore client configured.")
print("Collection:", ASTRA_DB_COLLECTION_NAME)

AstraDBVectorStore client configured.
Collection: gemini_groq_documents


## 7. Create the instructor's ten sample documents

A LangChain `Document` contains `page_content` (the text to embed and retrieve) and `metadata` (structured fields that can be filtered, such as `source`). The ten examples below preserve the instructor's topic mix and `tweet` / `news` / `website` labels.

**Why metadata?** Similarity alone may return a semantically close record from the wrong content category. Metadata filtering restricts candidates to the category your application permits.

In [30]:
document_1 = Document(
    page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "tweet"},
)
document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
    metadata={"source": "news"},
)
document_3 = Document(
    page_content="Building an exciting new project with LangChain - come check it out!",
    metadata={"source": "tweet"},
)
document_4 = Document(
    page_content="Robbers broke into the city bank and stole $1 million in cash.",
    metadata={"source": "news"},
)
document_5 = Document(
    page_content="Wow! That was an amazing movie. I can't wait to see it again.",
    metadata={"source": "tweet"},
)
document_6 = Document(
    page_content="Is the new iPhone worth the price? Read this review to find out.",
    metadata={"source": "website"},
)
document_7 = Document(
    page_content="The top 10 soccer players in the world right now.",
    metadata={"source": "website"},
)
document_8 = Document(
    page_content="LangGraph is the best framework for building stateful, agentic applications!",
    metadata={"source": "tweet"},
)
document_9 = Document(
    page_content="The stock market is down 500 points today due to fears of a recession.",
    metadata={"source": "news"},
)
document_10 = Document(
    page_content="I have a bad feeling I am going to get deleted :(",
    metadata={"source": "tweet"},
)

documents = [
    document_1, document_2, document_3, document_4, document_5,
    document_6, document_7, document_8, document_9, document_10,
]
print("Documents:", len(documents))
print("First document:", documents[0])

Documents: 10
First document: page_content='I had chocolate chip pancakes and scrambled eggs for breakfast this morning.' metadata={'source': 'tweet'}


## 8. Insert documents into the remote vector database

**What happens during insertion:** LangChain calls `embed_documents()` to create each vector, then stores the text, vector, ID and metadata in Astra DB.

**Why stable IDs?** Re-running this cell with the same IDs replaces the same records rather than blindly generating a new ID for every run. This makes classroom reruns easier to reason about. Do not reuse these IDs for unrelated content in a shared collection.

This performs real remote writes and may incur API/usage costs. The collection is not deleted automatically.

In [31]:
document_ids = [f"astra_lesson_doc_{i}" for i in range(1, len(documents) + 1)]
inserted_ids = vector_store.add_documents(
    documents=documents, ids=document_ids)
print(f"Submitted {len(inserted_ids)} document IDs to Astra DB.")
print("First three IDs:", inserted_ids[:3])

Submitted 10 document IDs to Astra DB.
First three IDs: ['astra_lesson_doc_1', 'astra_lesson_doc_2', 'astra_lesson_doc_3']


## 9. Basic semantic search

The instructor's first query is **"What is the weather"**. `similarity_search()` embeds the query, searches the remote collection, and returns the top matching `Document` objects.

This is semantic search, not exact keyword matching. The result is approximate relevance; inspect its text and metadata rather than assuming that the first result is always correct.

In [32]:
weather_results = vector_store.similarity_search("What is the weather", k=4)
for rank, doc in enumerate(weather_results, start=1):
    print(f"{rank}. {doc.page_content}")
    print("   metadata:", doc.metadata)

1. The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.
   metadata: {'source': 'news'}
2. The stock market is down 500 points today due to fears of a recession.
   metadata: {'source': 'news'}
3. Is the new iPhone worth the price? Read this review to find out.
   metadata: {'source': 'website'}
4. I have a bad feeling I am going to get deleted :(
   metadata: {'source': 'tweet'}


## 10. Search with metadata filtering

This preserves the instructor's query about LangChain abstractions and restricts the candidate documents to `source="tweet"`. `k=3` asks for at most three matches **after the filter is applied**.

Metadata filters are useful for access control and category restrictions, but the filter itself must be derived from trusted application logic in real systems—not blindly accepted from a user's prompt.

In [13]:
filtered_results = vector_store.similarity_search(
    "LangChain provides abstractions to make working with LLMs easy",
    k=3,
    filter={"source": "tweet"},
)
for doc in filtered_results:
    print(f'* "{doc.page_content}", metadata={doc.metadata}')

* "Building an exciting new project with LangChain - come check it out!", metadata={'source': 'tweet'}
* "LangGraph is the best framework for building stateful, agentic applications!", metadata={'source': 'tweet'}
* "I have a bad feeling I am going to get deleted :(", metadata={'source': 'tweet'}


## 11. Search with a score

`similarity_search_with_score()` returns `(Document, score)` pairs. The score is supplied by the Astra DB integration/backend. Do **not** call it a percentage or compare thresholds across unrelated metrics/models without checking the integration's score semantics.

Why inspect it? Scores help debug ranking and choose a threshold from observed examples. A high score does not prove that a document is factually correct.

In [14]:
scored_results = vector_store.similarity_search_with_score(
    "LangChain provides abstractions to make working with LLMs easy",
    k=3,
    filter={"source": "tweet"},
)
for doc, score in scored_results:
    print(f"score={score:.4f} | {doc.page_content!r} | metadata={doc.metadata}")

score=0.8922 | 'Building an exciting new project with LangChain - come check it out!' | metadata={'source': 'tweet'}
score=0.8619 | 'LangGraph is the best framework for building stateful, agentic applications!' | metadata={'source': 'tweet'}
score=0.7801 | 'I have a bad feeling I am going to get deleted :(' | metadata={'source': 'tweet'}


## 12. Convert the vector store into a retriever

A **retriever** is an abstraction that accepts a query and returns documents. It lets the rest of an application call `.invoke(question)` without depending directly on the vector database's concrete API.

First, use a similarity retriever with a top-k setting. The default search type is similarity, and `search_kwargs` passes options such as `k` to the underlying vector store.

In [15]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3},
)
retrieved_docs = retriever.invoke("Stealing from the bank is a crime")
for doc in retrieved_docs:
    print(doc.page_content, "|", doc.metadata)

Robbers broke into the city bank and stole $1 million in cash. | {'source': 'news'}
I have a bad feeling I am going to get deleted :( | {'source': 'tweet'}
The stock market is down 500 points today due to fears of a recession. | {'source': 'news'}


## 13. Similarity score threshold retriever

This mirrors the instructor's next example. It asks for the top candidate(s) and applies a score threshold of `0.5` through LangChain's threshold-retriever interface.

**Important:** a threshold is not a magic universal number. Before using it in a product, inspect the actual scores and relevant/irrelevant examples for this collection and integration version. If a backend's score mapping differs, recalibrate the threshold. We also pass the metadata filter `source="news"` when invoking it.

In [16]:
threshold_retriever = vector_store.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": 1, "score_threshold": 0.5},
)
threshold_docs = threshold_retriever.invoke(
    "Stealing from the bank is a crime",
    filter={"source": "news"},
)
for doc in threshold_docs:
    print(doc.page_content, "|", doc.metadata)

Robbers broke into the city bank and stole $1 million in cash. | {'source': 'news'}


## 14. Maximum Marginal Relevance (MMR) retriever

MMR selects results using both query relevance and diversity among the selected documents. It is helpful when several top results say nearly the same thing.

This is **not hybrid search**. Hybrid search combines different retrieval signals (for example, dense vector search and lexical/BM25 search); MMR is a diversity-aware selection strategy. The instructor transcript mentions MMR as a future topic, and the supplied notebook contains an MMR retriever example, so it is retained here with that distinction clarified.

In [17]:
mmr_retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 2, "fetch_k": 5},
)
mmr_docs = mmr_retriever.invoke(
    "Stealing from the bank is a crime",
    filter={"source": "news"},
)
for doc in mmr_docs:
    print(doc.page_content, "|", doc.metadata)

Robbers broke into the city bank and stole $1 million in cash. | {'source': 'news'}
The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees. | {'source': 'news'}


## 15. Groq — generation provider setup only

Gemini's job in this lesson is embedding. **Groq's job is text generation** for the optional RAG extension. We configure Groq in its own section so students can test and understand it independently from Gemini and Astra DB.

The model ID below is sent to Groq's API; the `openai/` prefix is part of the model identifier and does not mean this notebook calls the OpenAI API. Model availability can change, so replace the ID with a currently available model in your Groq console if needed.

In [18]:
from langchain_groq import ChatGroq

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY:
    raise ValueError("Set GROQ_API_KEY in .env before running this cell.")

groq_llm = ChatGroq(
    model="openai/gpt-oss-120b",
    api_key=GROQ_API_KEY,
    temperature=0,
)
print("Groq chat model configured.")

Groq chat model configured.


### 15.1 Test Groq independently

**Why:** A direct generation test checks Groq authentication/model access without involving retrieval. If this fails but Gemini and Astra search work, debug the Groq provider cell rather than the vector database.

In [19]:
groq_smoke_test = groq_llm.invoke(
    "Explain a vector database in one simple sentence.")
print(groq_smoke_test.content)

A vector database stores and indexes high‑dimensional numeric vectors so you can quickly find similar items (like text, images, or embeddings) by measuring their distance.


## 16. Optional extension — connect Astra retrieval to Groq with LCEL

This is the first section where the components are intentionally connected. The retriever fetches evidence from Astra DB; the prompt tells Groq to answer from that evidence; `StrOutputParser` returns plain text.

**What is LCEL?** LangChain Expression Language composes runnable components with the `|` operator. `RunnablePassthrough` forwards the original question, while `RunnableLambda` formats retrieved `Document` objects as context.

The chain does not make the underlying evidence perfect. Inspect retrieved documents and evaluate answer quality for representative questions before deployment.

In [20]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

retriever_for_rag = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3},
)


def format_docs(docs: Sequence[Document]) -> str:
    """Format retrieved documents as labelled context for the answer prompt."""
    return "\n\n".join(
        f"Source: {doc.metadata.get('source', 'unknown')}\n{doc.page_content}"
        for doc in docs
    )


rag_prompt = ChatPromptTemplate.from_template(
    """You are a careful assistant. Answer the question using only the context below.
If the context does not contain enough information, say that you do not have enough information.
Do not invent facts.

Context:
{context}

Question: {question}
Answer:"""
)

rag_chain = (
    {
        "context": retriever_for_rag | RunnableLambda(format_docs),
        "question": RunnablePassthrough(),
    }
    | rag_prompt
    | groq_llm
    | StrOutputParser()
)
print("LCEL RAG chain constructed.")

LCEL RAG chain constructed.


### 16.1 Run an end-to-end RAG question

This question checks whether Astra DB retrieves the weather-related sample record and Groq uses it to answer. If the answer is unsupported or the wrong source is retrieved, inspect the retrieval results first; a generation prompt cannot fully compensate for bad retrieval.

In [21]:
question = "What does the sample dataset say about tomorrow's weather?"
answer = rag_chain.invoke(question)
print("Question:", question)
print("Answer:", answer)
print("\nRetrieved context:")
for doc in retriever_for_rag.invoke(question):
    print(f"- ({doc.metadata.get('source', 'unknown')}) {doc.page_content}")

Question: What does the sample dataset say about tomorrow's weather?
Answer: The sample dataset indicates that tomorrow’s weather will be cloudy and overcast, with a high temperature of 62 degrees.

Retrieved context:
- (news) The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.
- (tweet) I have a bad feeling I am going to get deleted :(
- (tweet) I had chocolate chip pancakes and scrambled eggs for breakfast this morning.


## 17. Troubleshooting and production cautions

- **Authentication error:** verify the endpoint/token or provider key in the correct environment; never print secrets while debugging.
- **Collection dimension mismatch:** use the same embedding model and dimension for document and query vectors. Create a new collection if changing embedding spaces.
- **Rerun duplicates:** reuse stable IDs as this lesson does. For shared data, pick IDs/namespaces intentionally.
- **No results after threshold filtering:** inspect scores and reduce/recalibrate the threshold; verify the filter matches the actual metadata values.
- **MMR looks different:** it selects a relevance/diversity trade-off and may not return the same ranking as plain similarity.
- **Stale or wrong data:** check the collection name, namespace and source metadata. This notebook does not automatically delete a remote collection.
- **Security:** do not commit `.env`, Colab secrets, application tokens or outputs containing credentials. Use a minimally scoped token and rotate any token that was exposed.
- **Costs and limits:** Gemini embedding calls and Groq generation calls use provider quotas; Astra DB writes and reads use database resources. Check the current account pricing and limits before large runs.

### Interview-ready definitions

- **Embedding:** a numeric representation of content produced by a learned model.
- **Vector database:** a database that stores vectors and supports similarity retrieval, often alongside metadata.
- **Vector store:** a library abstraction for adding, embedding and searching documents in a vector backend.
- **Metadata filter:** a structured predicate restricting the candidate records (for example, `source="news"`).
- **Retriever:** an interface that maps a query to relevant documents.
- **Similarity threshold:** a cutoff intended to filter results below a chosen relevance level; calibration depends on the backend's scoring behavior.
- **MMR:** Maximal Marginal Relevance, a selection strategy balancing query relevance with diversity among results.
- **RAG:** Retrieval-Augmented Generation; retrieve evidence first, then condition a language model's answer on that evidence.

### Official references

- DataStax Astra DB + LangChain: https://docs.datastax.com/en/astra-db-serverless/integrations/langchain.html
- AstraDBVectorStore API: https://reference.langchain.com/python/langchain-astradb/vectorstores/AstraDBVectorStore
- Astra DB search with scores: https://reference.langchain.com/python/langchain-astradb/vectorstores/AstraDBVectorStore/similarity_search_with_score
- Gemini embeddings: https://ai.google.dev/gemini-api/docs/embeddings
- Groq LangChain integration: https://python.langchain.com/docs/integrations/chat/groq/

## 18. Self-check — explain before moving on

1. Why do we implement both `embed_documents()` and `embed_query()`?
2. Why must the embedding model and dimensions remain compatible with the collection?
3. What is the difference between `similarity_search()` and `as_retriever()`?
4. What does `filter={"source": "tweet"}` do?
5. Why should a score not automatically be described as a percent?
6. How is MMR different from hybrid search?
7. Which component calls Gemini, which component calls Astra DB, and which component calls Groq?
8. What is the main benefit and limitation of adding Groq after retrieval?

You are ready to move on when you can answer these questions and trace one query through the system: question → Gemini query embedding → Astra DB retrieval → retrieved context → Groq response.